## EDA mới: toàn bộ `train` / `valid` / `test` của TIB — chưa thực thi

Notebook này chạy độc lập trên Colab và thay thế phạm vi EDA của cohort 20/70 bài `Lecture` cũ. Chọn **Runtime → Run all**. Phần này đọc mọi dòng ở ba split của creator TIB tại revision đã ghim, không lọc theo `genre`, DOI, tác giả hoặc kích thước mẫu. Khóa từng dòng là `(split, source_index)`; bản ghi thiếu DOI hoặc URL vẫn ở ledger. Mã EDA được nhúng để notebook tải lên Colab có thể chạy độc lập, còn receipt/CSV được lưu vào Drive.

**Ranh giới bằng chứng:** phần mới sẽ đọc nội dung và abstract của `test`; sau khi chạy, split `test` của creator không còn là holdout chưa xem. Nó kiểm tra transcript, timestamp, abstract và metadata keyframe của mọi bản ghi; nó không tải/giải mã video, audio hoặc pixel keyframe và không tự xác nhận abstract là ground truth.


In [ ]:
import sys, subprocess, json, hashlib, types
from pathlib import Path
try:
    import datasets, pandas, matplotlib
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'datasets', 'pandas', 'matplotlib'])
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/RQ2_TIB_ALL_SPLITS_EDA')
OUT.mkdir(parents=True, exist_ok=True)
EDA_SOURCE = '"""Validation-only metadata scan for the RQ2 TIB text diagnostic."""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport re\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\n\nfrom datasets import load_dataset\n\n\nDATASET = "gigant/tib"\nREVISION = "e6ab9fd7c6acd164f3284b8c4b17eb9d94681eee"\nSPLIT = "valid"\nALL_SPLITS = ("train", "valid", "test")\n\n\ndef _word_count(value: object) -> int:\n    return len(str(value or "").split())\n\n\ndef _normalized(value: object) -> str:\n    return " ".join(str(value or "").casefold().split())\n\n\ndef _doi_url_match(doi: object, url: object) -> bool | None:\n    doi_id = re.search(r"10\\.5446/(\\d+)", str(doi or ""))\n    url_id = re.search(r"/media/(\\d+)", str(url or ""))\n    return doi_id.group(1) == url_id.group(1) if doi_id and url_id else None\n\n\ndef _finite_time(value: object) -> bool:\n    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)\n\n\ndef _scan_split(split: str) -> list[dict]:\n    """Read one pinned creator split and retain compact metadata."""\n    rows = []\n    dataset = load_dataset(DATASET, split=split, revision=REVISION, streaming=True)\n    for index, record in enumerate(dataset):\n        segments = record.get("transcript_segments") or {}\n        starts = (segments.get("start") or []) if isinstance(segments, dict) else []\n        ends = (segments.get("end") or []) if isinstance(segments, dict) else []\n        texts = (segments.get("text") or []) if isinstance(segments, dict) else []\n        aligned = len(starts) == len(ends) == len(texts)\n        numeric_times = all(\n            _finite_time(start) and _finite_time(end)\n            for start, end in zip(starts, ends)\n        )\n        valid_times = aligned and numeric_times and all(\n            start >= 0 and end >= start for start, end in zip(starts, ends)\n        )\n        transcript_words = _word_count(record.get("transcript"))\n        segment_words = sum(_word_count(text) for text in texts)\n        ordered = valid_times and all(starts[i] >= starts[i - 1] for i in range(1, len(starts)))\n        first_start_inversion = next(\n            (\n                {\n                    "previous_segment": i - 1,\n                    "previous_start_seconds": starts[i - 1],\n                    "current_segment": i,\n                    "current_start_seconds": starts[i],\n                }\n                for i in range(1, len(starts))\n                if valid_times and starts[i] < starts[i - 1]\n            ),\n            None,\n        )\n        overlap_count = sum(starts[i] < ends[i - 1] for i in range(1, len(starts))) if ordered else None\n        rows.append(\n            {\n                "source_index": index,\n                "doi": str(record.get("doi") or "").strip(),\n                "title": str(record.get("title") or "").strip(),\n                "genre": str(record.get("genre") or "").strip(),\n                "subject": str(record.get("subject") or "").strip(),\n                "author": str(record.get("author") or "").strip(),\n                "release_year": str(record.get("release_year") or "").strip(),\n                "language": str(record.get("language") or "").strip(),\n                "source_url_present": bool(record.get("url")),\n                "doi_source_url_match": _doi_url_match(record.get("doi"), record.get("url")),\n                "video_url_present": bool(record.get("video_url")),\n                "transcript_words": transcript_words,\n                "segment_text_words": segment_words,\n                "segment_word_ratio": round(segment_words / transcript_words, 3) if transcript_words else None,\n                "abstract_words": _word_count(record.get("abstract")),\n                "segment_count": len(texts),\n                "segments_aligned": aligned,\n                "segments_valid_times": valid_times,\n                "segments_ordered": ordered,\n                "first_start_inversion": first_start_inversion,\n                "segment_overlaps": overlap_count,\n                "empty_segment_texts": sum(not str(text or "").strip() for text in texts),\n                "duration_minutes": max(ends) / 60 if valid_times and ends else None,\n            }\n        )\n    return rows\n\n\ndef scan_validation() -> list[dict]:\n    """Historical validation-only diagnostic; retain its original row shape."""\n    return _scan_split(SPLIT)\n\n\ndef scan_all_splits() -> list[dict]:\n    """Scan every creator TIB record in train, valid and test, without genre filtering."""\n    return [{"split": split, **row} for split in ALL_SPLITS for row in _scan_split(split)]\n\n\ndef build_full_validation_cohort(rows: list[dict], output: Path) -> dict:\n    """Audit every validation row and export a reproducible, non-genre-gated cohort.\n\n    A DOI is metadata, not a row key: duplicate and missing DOIs remain in the\n    structurally usable list when their transcript is valid, with audit flags.\n    """\n    if len({row["source_index"] for row in rows}) != len(rows):\n        raise ValueError("Duplicate source_index in validation scan")\n    doi_counts = Counter(str(row.get("doi") or "").strip() for row in rows)\n    manifest = []\n    for row in sorted(rows, key=lambda item: item["source_index"]):\n        doi = str(row.get("doi") or "").strip()\n        flags = []\n        if not doi:\n            flags.append("missing_doi")\n        elif doi_counts[doi] > 1:\n            flags.append("duplicate_doi")\n        if not row.get("source_url_present"):\n            flags.append("missing_source_url")\n        reasons = []\n        if row.get("transcript_words", 0) <= 0:\n            reasons.append("missing_transcript")\n        if row.get("segment_count", 0) < 2:\n            reasons.append("insufficient_segments")\n        if not row.get("segments_aligned"):\n            reasons.append("misaligned_segments")\n        if not row.get("segments_valid_times"):\n            reasons.append("invalid_segment_times")\n        elif not row.get("segments_ordered"):\n            reasons.append("unordered_segments")\n        manifest.append({**row, "usable": not reasons, "exclusion_reasons": reasons,\n                         "audit_flags": flags})\n    usable = [row for row in manifest if row["usable"]]\n    canonical = [{"source_index": row["source_index"], "doi": row["doi"]} for row in usable]\n    digest = hashlib.sha256(json.dumps(canonical, ensure_ascii=False, separators=(",", ":")).encode()).hexdigest()\n    receipt = {\n        "dataset": DATASET,\n        "revision": REVISION,\n        "split_read": SPLIT,\n        "test_content_accessed": False,\n        "selection": "Every valid row with structurally usable transcript, regardless of genre or DOI/URL flags; no abstract quality or label claim",\n        "source_count": len(manifest),\n        "usable_count": len(usable),\n        "excluded_count": len(manifest) - len(usable),\n        "usable_index_doi_sha256": digest,\n        "manifest": manifest,\n        "usable": canonical,\n    }\n    output.parent.mkdir(parents=True, exist_ok=True)\n    output.write_text(json.dumps(receipt, ensure_ascii=False, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n    return receipt\n\n\ndef build_full_dataset_cohort(rows: list[dict], output: Path) -> dict:\n    """Export all split rows and a separately marked text-usable cohort.\n\n    Access to `test` here is intentional for whole-dataset EDA. Its previous\n    sealed-final-test status cannot be claimed for this new analysis.\n    """\n    keys = [(row["split"], row["source_index"]) for row in rows]\n    if len(set(keys)) != len(keys):\n        raise ValueError("Duplicate split/source_index in TIB scan")\n    unknown = {row["split"] for row in rows} - set(ALL_SPLITS)\n    if unknown:\n        raise ValueError(f"Unknown TIB splits: {sorted(unknown)}")\n    doi_counts = Counter(str(row.get("doi") or "").strip() for row in rows)\n    manifest = []\n    for row in sorted(rows, key=lambda item: (ALL_SPLITS.index(item["split"]), item["source_index"])):\n        doi = str(row.get("doi") or "").strip()\n        flags = []\n        if not doi:\n            flags.append("missing_doi")\n        elif doi_counts[doi] > 1:\n            flags.append("duplicate_doi_across_dataset")\n        if not row.get("source_url_present"):\n            flags.append("missing_source_url")\n        reasons = []\n        if row.get("transcript_words", 0) <= 0:\n            reasons.append("missing_transcript")\n        if row.get("segment_count", 0) < 2:\n            reasons.append("insufficient_segments")\n        if not row.get("segments_aligned"):\n            reasons.append("misaligned_segments")\n        if not row.get("segments_valid_times"):\n            reasons.append("invalid_segment_times")\n        elif not row.get("segments_ordered"):\n            reasons.append("unordered_segments")\n        manifest.append({**row, "usable": not reasons, "exclusion_reasons": reasons,\n                         "audit_flags": flags})\n    usable = [{"split": row["split"], "source_index": row["source_index"], "doi": row["doi"]}\n              for row in manifest if row["usable"]]\n    split_counts = {\n        split: {\n            "source_count": sum(row["split"] == split for row in manifest),\n            "usable_count": sum(row["split"] == split and row["usable"] for row in manifest),\n        }\n        for split in ALL_SPLITS\n    }\n    cross_split_groups = {}\n    for field in ("doi", "title", "author"):\n        groups: dict[str, set[str]] = defaultdict(set)\n        for row in manifest:\n            key = _normalized(row.get(field))\n            if key:\n                groups[key].add(row["split"])\n        cross_split_groups[field] = sum(len(splits) > 1 for splits in groups.values())\n    digest = hashlib.sha256(json.dumps(usable, ensure_ascii=False, separators=(",", ":")).encode()).hexdigest()\n    receipt = {\n        "dataset": DATASET,\n        "revision": REVISION,\n        "splits_read": list(ALL_SPLITS),\n        "test_content_accessed": True,\n        "final_test_holdout_sealed": False,\n        "selection": "Every creator record; usable denotes timed transcript structure only, regardless of genre or DOI/URL flags",\n        "source_count": len(manifest),\n        "usable_count": len(usable),\n        "excluded_count": len(manifest) - len(usable),\n        "split_counts": split_counts,\n        "cross_split_shared_group_counts": cross_split_groups,\n        "usable_split_index_doi_sha256": digest,\n        "manifest": manifest,\n        "usable": usable,\n    }\n    output.parent.mkdir(parents=True, exist_ok=True)\n    output.write_text(json.dumps(receipt, ensure_ascii=False, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n    return receipt\n\n\ndef select_diagnostic(rows: list[dict], limit: int = 20) -> list[dict]:\n    """Stable development selection; no test content or reference-based screen."""\n    candidates = [\n        row\n        for row in rows\n        if row["genre"].casefold() == "lecture"\n        and row["doi"]\n        and row["source_url_present"]\n        and row["transcript_words"] > 0\n        and row["segment_count"] >= 2\n        and row["segments_valid_times"]\n        and row["segments_ordered"]\n    ]\n    candidates.sort(key=lambda row: (hashlib.sha256(row["doi"].encode()).hexdigest(), row["doi"]))\n    selected = []\n    used_authors = set()\n    for row in candidates:\n        author_key = _normalized(row["author"])\n        if author_key and author_key in used_authors:\n            continue\n        selected.append(row)\n        if author_key:\n            used_authors.add(author_key)\n        if len(selected) == limit:\n            break\n    return selected\n\n\ndef write_receipt(rows: list[dict], selected: list[dict], output: Path) -> dict:\n    output.parent.mkdir(parents=True, exist_ok=True)\n    selected_ids = [row["doi"] for row in selected]\n    digest = hashlib.sha256(json.dumps(selected_ids, ensure_ascii=False).encode()).hexdigest()\n    receipt = {\n        "dataset": DATASET,\n        "revision": REVISION,\n        "split_read": SPLIT,\n        "test_content_accessed": False,\n        "selection": "SHA-256 DOI order, first 20 validation Lecture records with DOI, source URL and ordered timed transcript, one record per exact normalized author string; license is not an eligibility gate",\n        "source_count": len(rows),\n        "lecture_count": sum(row["genre"].casefold() == "lecture" for row in rows),\n        "candidate_count": sum(\n            row["genre"].casefold() == "lecture"\n            and row["doi"]\n            and row["source_url_present"]\n            and row["transcript_words"] > 0\n            and row["segment_count"] >= 2\n            and row["segments_valid_times"]\n            and row["segments_ordered"]\n            for row in rows\n        ),\n        "lecture_anomalies": [\n            {\n                "doi": row["doi"],\n                "doi_source_url_match": row["doi_source_url_match"],\n                "segments_ordered": row["segments_ordered"],\n                "first_start_inversion": row["first_start_inversion"],\n            }\n            for row in rows\n            if row["genre"].casefold() == "lecture"\n            and (row["doi_source_url_match"] is not True or not row["segments_ordered"])\n        ],\n        "selected_doi_sha256": digest,\n        "selected": selected,\n        "status": "LIMIT: text diagnostic candidate only; source/group/reference quality and human H2 gates pending",\n    }\n    output.write_text(json.dumps(receipt, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    return receipt\n\n\ndef scan_train_metadata_overlap(selected: list[dict]) -> dict:\n    """Project only DOI, title and author from train Parquet; never read test."""\n    import pyarrow.parquet as parquet\n    from huggingface_hub import HfFileSystem\n\n    selected_ids = {row["doi"] for row in selected if row["doi"]}\n    selected_titles: dict[str, set[str]] = defaultdict(set)\n    selected_authors: dict[str, set[str]] = defaultdict(set)\n    doi_indices: dict[str, set[int]] = defaultdict(set)\n    title_indices: dict[str, set[int]] = defaultdict(set)\n    author_indices: dict[str, set[int]] = defaultdict(set)\n    for row in selected:\n        source_index = row.get("source_index")\n        if isinstance(source_index, int):\n            if row["doi"]:\n                doi_indices[row["doi"]].add(source_index)\n            if row["title"]:\n                title_indices[_normalized(row["title"])].add(source_index)\n            if row["author"]:\n                author_indices[_normalized(row["author"])].add(source_index)\n        if row["title"]:\n            selected_titles[_normalized(row["title"])].add(row["doi"])\n        if row["author"]:\n            selected_authors[_normalized(row["author"])].add(row["doi"])\n    doi_hits: set[str] = set()\n    title_hits: set[str] = set()\n    author_counts = {doi: 0 for doi in selected_ids}\n    doi_index_hits: set[int] = set()\n    title_index_hits: set[int] = set()\n    author_index_counts = {index: 0 for row in selected\n                           if isinstance(index := row.get("source_index"), int)}\n    scanned = 0\n    filesystem = HfFileSystem()\n    for shard in range(2):\n        remote = (\n            f"datasets/{DATASET}@{REVISION}/data/"\n            f"train-{shard:05d}-of-00002.parquet"\n        )\n        with filesystem.open(remote, "rb") as source:\n            table = parquet.ParquetFile(source)\n            for batch in table.iter_batches(columns=["doi", "title", "author"], batch_size=4096):\n                for record in batch.to_pylist():\n                    scanned += 1\n                    doi = str(record.get("doi") or "").strip()\n                    title = _normalized(record.get("title"))\n                    author = _normalized(record.get("author"))\n                    if doi in selected_ids:\n                        doi_hits.add(doi)\n                        doi_index_hits.update(doi_indices.get(doi, ()))\n                    title_hits.update(selected_titles.get(title, ()))\n                    title_index_hits.update(title_indices.get(title, ()))\n                    for selected_doi in selected_authors.get(author, ()):\n                        author_counts[selected_doi] += 1\n                    for source_index in author_indices.get(author, ()):\n                        author_index_counts[source_index] += 1\n    return {\n        "revision": REVISION,\n        "train_records_scanned": scanned,\n        "columns_read": ["doi", "title", "author"],\n        "test_content_accessed": False,\n        "selected_doi_in_train": sorted(doi_hits),\n        "selected_exact_title_in_train": sorted(title_hits),\n        "selected_author_train_record_counts": {doi: n for doi, n in author_counts.items() if n},\n        "selected_source_index_doi_in_train": sorted(doi_index_hits),\n        "selected_source_index_exact_title_in_train": sorted(title_index_hits),\n        "selected_source_index_author_train_record_counts": {\n            str(index): n for index, n in sorted(author_index_counts.items()) if n\n        },\n    }\n'
DETAIL_SOURCE = '"""Characterize only the declared creator-TIB validation records for RQ2 EDA."""\n\nfrom __future__ import annotations\n\nimport re\nimport math\n\nfrom datasets import load_dataset\n\nDATASET = "gigant/tib"\nREVISION = "e6ab9fd7c6acd164f3284b8c4b17eb9d94681eee"\nALL_SPLITS = ("train", "valid", "test")\n\n\ndef _tokens(value: object) -> list[str]:\n    return re.findall(r"\\b\\w+\\b", str(value or "").casefold())\n\n\ndef _preview(value: object, limit: int = 180) -> str:\n    return " ".join(str(value or "").split())[:limit]\n\n\ndef scan_selected_details(selected: list[dict]) -> list[dict]:\n    """Read the pinned `valid` split; retain metadata and short source examples only."""\n    wanted = {row["doi"] for row in selected}\n    found: dict[str, dict] = {}\n    rows = load_dataset(DATASET, split="valid", revision=REVISION, streaming=True)\n    for record in rows:\n        doi = str(record.get("doi") or "").strip()\n        if doi not in wanted:\n            continue\n        segments = record.get("transcript_segments") or {}\n        texts = list(segments.get("text") or [])\n        starts = [float(value) for value in segments.get("start") or []]\n        ends = [float(value) for value in segments.get("end") or []]\n        if not texts or not (len(texts) == len(starts) == len(ends)):\n            raise ValueError(f"Misaligned transcript segments for {doi}")\n        keyframes = record.get("keyframes") or {}\n        slides = list(keyframes.get("slide") or [])\n        frames = list(keyframes.get("frames") or [])\n        ranges = list(keyframes.get("timestamp") or [])\n        keyframe_arrays_aligned = len(slides) == len(frames) == len(ranges)\n        keyframe_intervals_ordered = keyframe_arrays_aligned and all(\n            len(pair) == 2 and 0 <= float(pair[0]) <= float(pair[1])\n            for pair in ranges\n        ) and all(float(ranges[index][0]) >= float(ranges[index - 1][0])\n                  for index in range(1, len(ranges)))\n        duration = max(ends)\n        abstract = str(record.get("abstract") or "")\n        segment_tokens = _tokens(" ".join(texts))\n        source_tokens = _tokens(record.get("transcript"))\n        license_text = str(record.get("license") or "")\n        found[doi] = {\n            "doi": doi,\n            "title": str(record.get("title") or "").strip(),\n            "subject": str(record.get("subject") or "").strip(),\n            "language": str(record.get("language") or "").strip(),\n            "release_year": str(record.get("release_year") or "").strip(),\n            "source_url": str(record.get("url") or "").strip(),\n            "video_url_present": bool(record.get("video_url")),\n            "license_label": license_text.splitlines()[0].strip() if license_text else "",\n            "transcript_words": len(str(record.get("transcript") or "").split()),\n            "segment_text_words": sum(len(str(text or "").split()) for text in texts),\n            "whitespace_word_count_matches": len(str(record.get("transcript") or "").split()) == sum(\n                len(str(text or "").split()) for text in texts\n            ),\n            "segment_count": len(texts),\n            "duration_minutes": round(duration / 60, 2),\n            "words_per_minute": round(len(source_tokens) / (duration / 60), 1) if duration else None,\n            "transcript_matches_segments": source_tokens == segment_tokens,\n            "zero_duration_segments": sum(start == end for start, end in zip(starts, ends)),\n            "gaps_over_30_sec": sum(starts[index] - ends[index - 1] > 30\n                                    for index in range(1, len(starts))),\n            "abstract_words": len(abstract.split()),\n            "abstract_chars": len(abstract),\n            "abstract_text": abstract,\n            "abstract_preview": _preview(abstract),\n            "transcript_example": _preview(texts[len(texts) // 2]),\n            "keyframe_count": len(ranges),\n            "first_keyframe_filename": str(slides[0]) if slides else "",\n            "first_keyframe_seconds": [round(float(value), 2) for value in ranges[0]] if ranges else [],\n            "keyframe_arrays_aligned": keyframe_arrays_aligned,\n            "keyframe_intervals_ordered": keyframe_intervals_ordered,\n            "keyframe_last_minus_transcript_end_sec": round(float(ranges[-1][1]) - duration, 2)\n            if ranges else None,\n            "keyframe_pixels_read": False,\n            "audio_sample_read": False,\n        }\n    missing = wanted - found.keys()\n    if missing:\n        raise RuntimeError(f"Selected validation DOI missing: {sorted(missing)}")\n    return [found[row["doi"]] for row in selected]\n\n\ndef _scan_usable_details_for_split(usable: list[dict], split: str) -> list[dict]:\n    """Characterize all eligible validation source indices; log item failures.\n\n    DOI is carried for traceability but never used as the unique lookup key.\n    Text/abstract properties are descriptive, not ground-truth judgments.\n    """\n    wanted = {row["source_index"]: row for row in usable}\n    if len(wanted) != len(usable):\n        raise ValueError("Duplicate source_index in usable cohort")\n    found: dict[int, dict] = {}\n    rows = load_dataset(DATASET, split=split, revision=REVISION, streaming=True)\n    for source_index, record in enumerate(rows):\n        if source_index not in wanted:\n            continue\n        doi = str(record.get("doi") or "").strip()\n        expected_doi = wanted[source_index]["doi"]\n        try:\n            if doi != expected_doi:\n                raise ValueError(f"DOI changed from {expected_doi!r} to {doi!r}")\n            segments = record.get("transcript_segments") or {}\n            if not isinstance(segments, dict):\n                raise ValueError("transcript_segments is not a mapping")\n            texts = list(segments.get("text") or [])\n            starts = list(segments.get("start") or [])\n            ends = list(segments.get("end") or [])\n            if len(texts) < 2 or len(texts) != len(starts) or len(texts) != len(ends):\n                raise ValueError("transcript segment arrays are missing or misaligned")\n            if not all(isinstance(value, (int, float)) and not isinstance(value, bool)\n                       and math.isfinite(value) for value in starts + ends):\n                raise ValueError("transcript segment times are not finite numbers")\n            if any(start < 0 or end < start for start, end in zip(starts, ends)):\n                raise ValueError("transcript segment time range is invalid")\n            if any(starts[index] < starts[index - 1] for index in range(1, len(starts))):\n                raise ValueError("transcript segments are unordered")\n            keyframes = record.get("keyframes") or {}\n            if not isinstance(keyframes, dict):\n                raise ValueError("keyframes is not a mapping")\n            slides = list(keyframes.get("slide") or [])\n            frames = list(keyframes.get("frames") or [])\n            ranges = list(keyframes.get("timestamp") or [])\n            keyframe_arrays_aligned = len(slides) == len(frames) == len(ranges)\n            valid_ranges = []\n            for pair in ranges:\n                valid_ranges.append(\n                    isinstance(pair, (list, tuple)) and len(pair) == 2\n                    and all(isinstance(value, (int, float)) and not isinstance(value, bool)\n                            and math.isfinite(value) for value in pair)\n                    and 0 <= pair[0] <= pair[1]\n                )\n            ordered_keyframes = (keyframe_arrays_aligned and all(valid_ranges)\n                                 and all(ranges[index][0] >= ranges[index - 1][0]\n                                         for index in range(1, len(ranges))))\n            transcript = str(record.get("transcript") or "")\n            abstract = str(record.get("abstract") or "")\n            license_text = str(record.get("license") or "")\n            segment_text = " ".join(str(text or "") for text in texts)\n            duration = max(ends)\n            found[source_index] = {\n                "source_index": source_index,\n                "doi": doi,\n                "status": "ok",\n                "genre": str(record.get("genre") or "").strip(),\n                "language": str(record.get("language") or "").strip(),\n                "source_url": str(record.get("url") or "").strip(),\n                "video_url_present": bool(record.get("video_url")),\n                "license_label": license_text.splitlines()[0].strip() if license_text else "",\n                "transcript_words": len(transcript.split()),\n                "segment_text_words": len(segment_text.split()),\n                "transcript_matches_segments": _tokens(transcript) == _tokens(segment_text),\n                "segment_count": len(texts),\n                "duration_minutes": round(duration / 60, 2),\n                "words_per_minute": round(len(_tokens(transcript)) / (duration / 60), 1) if duration else None,\n                "zero_duration_segments": sum(start == end for start, end in zip(starts, ends)),\n                "gaps_over_30_sec": sum(starts[index] - ends[index - 1] > 30\n                                        for index in range(1, len(starts))),\n                "abstract_words": len(abstract.split()),\n                "abstract_chars": len(abstract),\n                "abstract_text": abstract,\n                "abstract_preview": _preview(abstract),\n                "transcript_example": _preview(texts[len(texts) // 2]),\n                "keyframe_count": len(ranges),\n                "keyframe_arrays_aligned": keyframe_arrays_aligned,\n                "keyframe_intervals_ordered": ordered_keyframes,\n                "keyframe_last_minus_transcript_end_sec": round(ranges[-1][1] - duration, 2)\n                if ranges and valid_ranges[-1] else None,\n                "keyframe_pixels_read": False,\n                "audio_sample_read": False,\n            }\n        except (TypeError, ValueError, OverflowError, AttributeError, IndexError) as exc:\n            found[source_index] = {\n                "source_index": source_index,\n                "doi": doi,\n                "status": "error",\n                "error_type": type(exc).__name__,\n                "error": str(exc),\n            }\n    for source_index, row in wanted.items():\n        if source_index not in found:\n            found[source_index] = {\n                "source_index": source_index,\n                "doi": row["doi"],\n                "status": "missing",\n                "error": "Validation source index absent at pinned revision",\n            }\n    return [found[index] for index in sorted(wanted)]\n\n\ndef scan_all_usable_details(usable: list[dict]) -> list[dict]:\n    """Validation-only detail scan retained for the earlier extension."""\n    return _scan_usable_details_for_split(usable, "valid")\n\n\ndef scan_all_dataset_details(usable: list[dict]) -> list[dict]:\n    """Inspect all structurally usable rows across every creator split."""\n    unknown = {row.get("split") for row in usable} - set(ALL_SPLITS)\n    if unknown:\n        raise ValueError(f"Unknown TIB splits: {sorted(map(str, unknown))}")\n    details = []\n    for split in ALL_SPLITS:\n        rows = [row for row in usable if row["split"] == split]\n        details.extend({"split": split, **detail}\n                       for detail in _scan_usable_details_for_split(rows, split))\n    return details\n'
eda_ns = {'__name__': 'tib_all_splits_eda'}
detail_ns = {'__name__': 'tib_all_splits_detail_eda'}
exec(compile(EDA_SOURCE, 'rq2-tib-valid-eda.py', 'exec'), eda_ns)
exec(compile(DETAIL_SOURCE, 'rq2_tib_valid_detail_eda.py', 'exec'), detail_ns)
assert eda_ns['REVISION'] == detail_ns['REVISION']
print('Pinned TIB revision:', eda_ns['REVISION'])
print('Splits:', eda_ns['ALL_SPLITS'])


In [ ]:
all_rows = eda_ns['scan_all_splits']()
cohort = eda_ns['build_full_dataset_cohort'](all_rows, OUT/'all_splits_cohort.json')
assert cohort['splits_read'] == ['train', 'valid', 'test']
assert cohort['test_content_accessed'] is True and cohort['final_test_holdout_sealed'] is False
assert len(cohort['manifest']) == cohort['source_count'] == len(all_rows)
assert len({(row['split'], row['source_index']) for row in cohort['manifest']}) == len(all_rows)
expected_counts = {'train': 7282, 'valid': 910, 'test': 911}
observed_counts = {split: value['source_count'] for split, value in cohort['split_counts'].items()}
print('Observed source rows:', observed_counts, '| documented:', expected_counts)
print('All rows:', cohort['source_count'], '| text-usable:', cohort['usable_count'], '| structurally unusable:', cohort['excluded_count'])
print('Usable cohort SHA-256:', cohort['usable_split_index_doi_sha256'])
manifest_df = pd.DataFrame(cohort['manifest'])
manifest_df.to_csv(OUT/'all_splits_manifest.csv', index=False)
display(pd.crosstab(manifest_df['split'], manifest_df['genre'], dropna=False))
display(pd.DataFrame(cohort['split_counts']).T)
if observed_counts != expected_counts:
    print('COUNT MISMATCH: inspect source revision/schema before treating EDA as complete')


In [ ]:
from collections import Counter
exclusion_counts = Counter(reason for reasons in manifest_df['exclusion_reasons'] for reason in reasons)
audit_flag_counts = Counter(flag for flags in manifest_df['audit_flags'] for flag in flags)
print('Transcript-structure exclusions:', dict(exclusion_counts))
print('DOI/source warnings:', dict(audit_flag_counts))
display(manifest_df.loc[~manifest_df['usable'], ['split','source_index','doi','genre','exclusion_reasons']].head(40))
for field in ['transcript_words','abstract_words','segment_count','duration_minutes']:
    print(field)
    display(manifest_df.groupby('split')[field].describe(percentiles=[.1,.25,.5,.75,.9]))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for split, group in manifest_df.groupby('split'):
    group['transcript_words'].clip(upper=30000).hist(ax=axes[0], bins=30, alpha=.45, label=split)
    group['abstract_words'].clip(upper=1000).hist(ax=axes[1], bins=30, alpha=.45, label=split)
    group['duration_minutes'].dropna().clip(upper=180).hist(ax=axes[2], bins=30, alpha=.45, label=split)
for ax, title in zip(axes, ['Transcript words (clipped for display)','Abstract words (clipped for display)','Minutes (clipped for display)']):
    ax.set_title(title); ax.legend()
fig.tight_layout(); plt.show()
print('Cross-split shared DOI/title/author groups:', cohort['cross_split_shared_group_counts'])
shared_rows = []
for field in ('doi', 'title', 'author'):
    keys = manifest_df[field].fillna('').astype(str).map(lambda value: ' '.join(value.casefold().split()))
    for key, group in manifest_df.assign(group_key=keys).loc[keys.ne('')].groupby('group_key'):
        if group['split'].nunique() > 1:
            for row in group.itertuples(index=False):
                shared_rows.append({'field':field,'normalized_value':key,'split':row.split,
                                    'source_index':int(row.source_index),'doi':row.doi})
shared_df = pd.DataFrame(shared_rows, columns=['field','normalized_value','split','source_index','doi'])
shared_df.to_csv(OUT/'cross_split_shared_groups.csv', index=False)
print('Cross-split rows requiring review:',len(shared_df))
display(shared_df.head(30))


In [ ]:
details = detail_ns['scan_all_dataset_details'](cohort['usable'])
assert len(details) == cohort['usable_count']
assert {(row['split'], row['source_index']) for row in details} == {(row['split'], row['source_index']) for row in cohort['usable']}
(OUT/'all_splits_details.json').write_text(json.dumps(details, ensure_ascii=False, indent=2, allow_nan=False)+'\n', encoding='utf-8')
detail_df = pd.DataFrame(details)
detail_df.to_csv(OUT/'all_splits_details.csv', index=False)
if len(detail_df):
    review_df = detail_df.reindex(columns=['split','source_index','doi','status','genre','source_url','license_label',
        'abstract_words','abstract_preview','transcript_words','transcript_matches_segments',
        'keyframe_count','keyframe_arrays_aligned','keyframe_intervals_ordered','video_url_present']).copy()
    review_df['reference_status'] = 'unverified_creator_abstract'
    review_df['media_status'] = 'not_decoded'
    review_df['short_abstract_under_30_words'] = pd.to_numeric(review_df['abstract_words'], errors='coerce').lt(30)
    review_df.to_csv(OUT/'reference_and_media_review_queue.csv', index=False)
    display(review_df.groupby('split', dropna=False)[['short_abstract_under_30_words']].sum())
    display(review_df.loc[review_df['short_abstract_under_30_words']].head(30))
detail_lookup = {(row['split'], row['source_index']): row for row in details}
manifest_df['detail_status'] = [detail_lookup.get((row['split'], row['source_index']), {}).get('status', 'not_run_unusable_transcript')
                                for _, row in manifest_df.iterrows()]
manifest_df.to_csv(OUT/'all_splits_per_record_audit.csv', index=False)
print('Detailed scan statuses:', manifest_df['detail_status'].value_counts().to_dict())
print('Per-split detail coverage:')
display(pd.crosstab(manifest_df['split'], manifest_df['detail_status']))
complete = observed_counts == expected_counts and detail_df['status'].eq('ok').all()
coverage = {
    'dataset': cohort['dataset'], 'revision': cohort['revision'],
    'splits_read': cohort['splits_read'], 'test_content_accessed': True,
    'final_test_holdout_sealed': False, 'expected_counts': expected_counts,
    'observed_counts': observed_counts, 'source_count': cohort['source_count'],
    'text_usable_count': cohort['usable_count'], 'detail_status_counts': manifest_df['detail_status'].value_counts().to_dict(),
    'source_rows_accounted_for': len(manifest_df), 'structural_eda_complete': bool(complete),
    'reference_ground_truth_qualified': False, 'media_pixels_or_audio_read': False,
    'evaluation_status': 'not_frozen', 'decision': 'LIMIT: source/reference/media review and scoreable cohort pending'
}
(OUT/'coverage.json').write_text(json.dumps(coverage, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
display(detail_df.loc[detail_df['status'].ne('ok')].reindex(columns=['split','source_index','doi','status','error']).head(40))
print(json.dumps(coverage, ensure_ascii=False, indent=2))


### Quyết định sau khi chạy

Mở `all_splits_per_record_audit.csv`, `cross_split_shared_groups.csv`, `reference_and_media_review_queue.csv` và `coverage.json` trên Drive. Mọi dòng nguồn phải được kiểm kê, mọi dòng text-usable phải có trạng thái chi tiết; các lỗi vẫn giữ trong ledger. Xét provenance và độ phù hợp của abstract theo từng dòng trước khi dùng ROUGE, xét nhóm trùng/chéo split và media riêng trước khi chốt cohort đánh giá. Nếu muốn một kết luận trên holdout chưa xem, cần nguồn độc lập mới vì EDA này đã đọc cả `test`. **Hiện các ô mới chưa có output; không có số lượng text-usable hay kết quả mô hình cho toàn bộ TIB.**
